# Day 11: Reddit Topic Index

**Goal**: Fetch Reddit posts, embed them with CLIP text encoder, and build a FAISS topic index for trend-aware matching.

**Note**: Requires Reddit API credentials (`REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET`).  
If you don't have them yet, a static fallback list is used automatically.

**Done checkpoint**:
- `data/embeddings/topic_index.faiss` and `topic_records.json` saved
- Given any image caption, system returns matching topic labels

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/PinterestDemo'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'CWD: {os.getcwd()}')

In [ ]:
!pip install -q torch torchvision transformers faiss-cpu Pillow pandas numpy tqdm pyyaml praw
print('Done.')

## 2. Configure Reddit Credentials

In [ ]:
from google.colab import userdata
import os

try:
    os.environ['REDDIT_CLIENT_ID']     = userdata.get('REDDIT_CLIENT_ID')
    os.environ['REDDIT_CLIENT_SECRET'] = userdata.get('REDDIT_CLIENT_SECRET')
    os.environ['REDDIT_USER_AGENT']    = 'PinterestDemo/1.0'
    print('Reddit credentials set from Colab Secrets.')
    USE_REDDIT = True
except Exception:
    print('Reddit credentials not set. Using static fallback topic list.')
    print('To get credentials: https://www.reddit.com/prefs/apps')
    USE_REDDIT = False

## 3. Load Config

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE = Path(PROJECT_DIR)
config['model']['device'] = 'cuda'
config['model']['clip_model'] = 'openai/clip-vit-base-patch32'

print('Config ready.')
print(f'Subreddits configured: {config["reddit"]["subreddits"]}')

## 4. Fetch Reddit Posts (or Use Static Fallback)

In [ ]:
from src.enrichment.reddit import RedditTopicFetcher, build_topic_records, TopicPost

if USE_REDDIT:
    fetcher = RedditTopicFetcher()
    posts = fetcher.fetch_all_subreddits(
        config['reddit']['subreddits'],
        posts_per_subreddit=config['reddit']['posts_per_subreddit']
    )
    print(f'Fetched {len(posts)} posts.')
else:
    # Static fallback topics — covers major image themes
    static_topics = [
        ('fashion', 'Best summer outfits 2025 - streetwear edition'),
        ('fashion', 'How to style wide-leg trousers for work'),
        ('fashion', 'Minimalist wardrobe essentials for men'),
        ('streetwear', 'New Adidas collection drops this week'),
        ('streetwear', 'Sneaker release dates and hype drops'),
        ('malefashionadvice', 'Business casual for tech office - what works'),
        ('femalefashionadvice', 'Transitional outfits from summer to fall'),
        ('femalefashionadvice', 'Workwear capsule wardrobe on a budget'),
        ('travel', 'Hidden gems in Southeast Asia you need to visit'),
        ('travel', 'Visiting Eiffel Tower - tips and travel photos'),
        ('travel', 'Best cafes in Tokyo - a complete photo guide'),
        ('travel', 'Budget backpacking Europe 2025 itinerary'),
        ('architecture', 'Most stunning modern buildings in the world'),
        ('architecture', 'Historic cathedrals and churches photography'),
        ('food', 'Street food tour of Istanbul - photos and tips'),
        ('food', 'Aesthetic coffee shop interior design'),
        ('sports', 'Best sports action photography techniques'),
        ('sports', 'Marathon running gear and training tips'),
        ('art', 'Contemporary art exhibition photography'),
        ('music', 'Concert photography tips and gear'),
    ]
    posts = [TopicPost(subreddit=s, title=t, score=1000, url='') for s, t in static_topics]
    print(f'Using {len(posts)} static fallback topics.')

records = build_topic_records(posts)
print(f'Topic records: {len(records)}')
print(f'Sample: {records[0]}')

## 5. Build Topic Index

In [ ]:
from src.models.clip_encoder import CLIPEncoder
from src.models.text_encoder import TextEncoder
from src.enrichment.topic_index import TopicIndex

encoder = CLIPEncoder(model_name=config['model']['clip_model'], device='cuda')
text_encoder = TextEncoder(clip_encoder=encoder)

topic_index = TopicIndex()
topic_index.build(records, text_encoder)

# Save to Drive
index_path  = str(BASE / 'data/embeddings/topic_index.faiss')
records_path = str(BASE / 'data/embeddings/topic_records.json')
topic_index.save(index_path=index_path, records_path=records_path)

print(f'Topic index saved: {topic_index.index.ntotal} topics.')

## 6. Test Topic Matching

In [ ]:
# Test: given a text description, what topics match?
test_queries = [
    'a person wearing a blue denim jacket and white sneakers',
    'the Eiffel Tower at night with city lights',
    'colorful street food market with spices and vegetables',
    'modern glass skyscraper in a downtown city',
    'woman in elegant evening dress at a gala event',
]

for query_text in test_queries:
    emb = text_encoder.encode_single(query_text)
    hits = topic_index.search(emb, top_k=3)
    labels = [f"r/{h['subreddit']} ({h['topic_score']:.3f})" for h in hits]
    print(f'Query: "{query_text[:50]}"')
    print(f'  Matched: {" | ".join(labels)}')
    print()

## 7. Test with Real Image Captions

In [ ]:
import pandas as pd

df = pd.read_csv(BASE / 'data/metadata/images.csv')
captioned = df[df['caption'].notna() & (df['caption'] != '')]

if len(captioned) > 0:
    print('Topic matching for real image captions:\n')
    for _, row in captioned.sample(min(5, len(captioned))).iterrows():
        caption = str(row['caption'])
        emb = text_encoder.encode_single(caption)
        hits = topic_index.search(emb, top_k=2)
        labels = topic_index.get_topic_labels(emb, top_k=2)
        print(f'  Caption: "{caption}"')
        print(f'  Topics:  {labels}')
        print()
else:
    print('No captioned images found. Run Day 8 notebook first.')

## ✅ Day 11 Done Checkpoint

In [ ]:
from pathlib import Path
from src.enrichment.topic_index import TopicIndex

ti = TopicIndex()
ti.load(index_path=index_path, records_path=records_path)

assert ti.index.ntotal > 0, 'Topic index is empty'

test_emb = text_encoder.encode_single('blue denim jacket street style')
hits = ti.search(test_emb, top_k=3)
assert len(hits) == 3

print('Day 11 COMPLETE')
print(f'  Topic index: {ti.index.ntotal} topics')
print(f'  Top match for test query: r/{hits[0]["subreddit"]} (score={hits[0]["topic_score"]:.3f})')